In [2]:
import os
import pyspark

# Autodetekcja wersji Sparka → właściwy connector
spark_version = pyspark.__version__
print(f"Wykryta wersja PySpark: {spark_version}")

if spark_version.startswith("4"):
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2"
else:
    # Spark 3.x
    KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

print(f"Użyty connector:        {KAFKA_PACKAGE}")

# Musi być ustawione PRZED SparkSession.builder
os.environ['PYSPARK_SUBMIT_ARGS'] = f'--packages {KAFKA_PACKAGE} pyspark-shell'

Wykryta wersja PySpark: 4.0.0.dev2
Użyty connector:        org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.0-preview2


In [4]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Lab4-Kafka")
    .config("spark.jars.packages", KAFKA_PACKAGE)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"Spark {spark.version} — gotowy")

Spark 4.0.0-preview2 — gotowy


In [5]:
kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

kafka_raw.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [6]:
def process_batch(df, batch_id, tstop=5):
    print(f"Batch ID: {batch_id}")
    df.show(truncate=False)

kafka_raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "transactions")
    .load()
)

query = (
    kafka_raw.writeStream
    .format("console")
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False)
    .start()
)

Batch ID: 0
+---+-----+-----+---------+------+---------+-------------+
|key|value|topic|partition|offset|timestamp|timestampType|
+---+-----+-----+---------+------+---------+-------------+
+---+-----+-----+---------+------+---------+-------------+

Batch ID: 1
+----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------------+---------+------+-----------------------+-------------+
|key |value                                                                                                                                                                                                         

In [7]:
query.stop()


In [8]:
from pyspark.sql.functions import col

df = kafka_raw.select(
    col("timestamp").alias("time"),
    col("value").cast("string")
)

query = (
    df.writeStream
    .format("console")
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False)
    .start()
)

In [9]:
query.stop()

In [11]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from pyspark.sql.functions import from_json, col

tx_schema = StructType([
    StructField("tx_id",     StringType()),
    StructField("user_id",   StringType()),
    StructField("amount",    DoubleType()),
    StructField("store",     StringType()),
    StructField("category",  StringType()),
    StructField("timestamp", StringType()),
])

step2 = kafka_raw.select(
    from_json(col("value").cast("string"), tx_schema).alias("tx")
)

query = (
    step2.writeStream
    .format("console")
    .outputMode("append")
    .foreachBatch(process_batch)
    .option("truncate", False)
    .start()
)

In [12]:
query.stop()

In [13]:
from pyspark.sql.functions import to_timestamp

df = (
    kafka_raw
    .select(from_json(col("value").cast("string"), tx_schema).alias("tx"))
    .select("tx.*")
    .withColumn("timestamp", to_timestamp("timestamp"))
)

print("Finalny schemat:")
df.printSchema()

Finalny schemat:
root
 |-- tx_id: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- store: string (nullable = true)
 |-- category: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [15]:
from pyspark.sql.functions import window, count, sum as _sum, round as _round

windowed = (
    df
    .withWatermark("timestamp", "3 seconds")
    .groupBy(window("timestamp", "20 seconds"), "store")
    .agg(
        count("tx_id").alias("liczba_tx"),
        _round(_sum("amount"), 2).alias("suma_PLN"),
    )
)

In [16]:
query = (
    windowed.writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .start()
)

In [17]:
query.stop()

In [19]:
from pyspark.sql.functions import to_json, struct, lit

alerts = (
    df
    .filter(col("amount") > 3000)
    .select(
        to_json(
            struct(
                "tx_id", "user_id", "amount", "store", "category",
                col("timestamp").cast("string"),
                lit("HIGH").alias("alert_level"),
            )
        ).alias("value")
    )
)

alert_query = (
    alerts.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "alerts")
    .outputMode("append")
    .start()
)

print("Strumień alertów uruchomiony.")

AnalysisException: checkpointLocation must be specified either through option("checkpointLocation", ...) or SparkSession.conf.set("spark.sql.streaming.checkpointLocation", ...).

In [20]:
from pyspark.sql.functions import to_json, struct, lit

alerts = (
    df
    .filter(col("amount") > 3000)
    .select(
        to_json(
            struct(
                "tx_id", "user_id", "amount", "store", "category",
                col("timestamp").cast("string"),
                lit("HIGH").alias("alert_level"),
            )
        ).alias("value")
    )
)

alert_query = (
    alerts.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("topic", "alerts")

    .option("checkpointLocation", "/tmp/checkpoints_alerts")

    .outputMode("append")
    .start()
)

print("Strumień alertów uruchomiony.")

Strumień alertów uruchomiony.


In [ ]:
# 1. Dlaczego w trybie 'append' pierwsze wyniki pojawiają się z opóźnieniem?
# append pokazuje wyniki dopiero po zamknięciu okna.
# Spark czeka na watermark i ewentualne spóźnione dane.

# 2. Co się stanie jeśli ustawisz watermark na "0 seconds"?
# Spark nie będzie czekał na spóźnione dane.
# Spóźnione rekordy zostaną odrzucone natychmiast.

# 3. Jaka jest różnica między window() w Lab 2 (batch) a window() tutaj (streaming)?
# W batch operujemy na skończonym zbiorze danych.
# W streamingu dane napływają cały czas i okna są aktualizowane dynamicznie.

# 4. Dlaczego do zapisu do Kafki używamy to_json(), a nie po prostu select("value")?
# Kafka przyjmuje dane jako string/bytes.
# to_json() zamienia struct Spark na JSON string.

In [22]:
alert_query.stop()
spark.stop()